# Phase 4: Multi-Version Compressed Neural Network Training (MVC-NNT)
**Paper Fig. 8, Eq. 12:** Train 3 compressed model versions on distinct dataset partitions.

### Key components:
1. **Dataset partitioning:** CIFAR-10 train split into D1, D2, D3
2. **Per-partition training:** Each expert fine-tuned on its own partition (adversarial)
3. **Random selection at inference:** Unpredictability defeats targeted attacks
4. **Gradient coherence analysis:** Verify diversity between model versions

In [1]:
import os, gc, time
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchattacks
import numpy as np
from torch.utils.data import Subset, DataLoader
from tqdm import tqdm

from utils import (
    device, evaluate, train_one_epoch,
    save_checkpoint, load_checkpoint, count_zero_params, freeze_zeros
)
from ensemble_utils import MVC_NNT

print(f'Device: {device}')



Device: cuda


In [2]:
# ==========================================================================
# DATASET PARTITIONING (Paper Fig. 8: D1, D2, D3)
# ==========================================================================
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])
transform_test = transforms.Compose([transforms.ToTensor()])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
testloader = DataLoader(testset, batch_size=100, shuffle=False, num_workers=2)

# Split into 3 non-overlapping partitions
np.random.seed(42)  # Reproducible split
indices = np.random.permutation(len(trainset))
split_size = len(trainset) // 3

D1_indices = indices[:split_size]
D2_indices = indices[split_size:2*split_size]
D3_indices = indices[2*split_size:]

D1_loader = DataLoader(Subset(trainset, D1_indices), batch_size=128, shuffle=True, num_workers=2)
D2_loader = DataLoader(Subset(trainset, D2_indices), batch_size=128, shuffle=True, num_workers=2)
D3_loader = DataLoader(Subset(trainset, D3_indices), batch_size=128, shuffle=True, num_workers=2)

eps, alpha, steps = 8/255, 2/255, 20

print(f'D1: {len(D1_indices)} samples')
print(f'D2: {len(D2_indices)} samples')
print(f'D3: {len(D3_indices)} samples')

D1: 16666 samples
D2: 16666 samples
D3: 16668 samples


In [3]:
# Load Phase 2 experts as starting points for MVC training
expert_configs = [
    {'ratio': 30, 'loader': D1_loader, 'name': 'MVC-30'},
    {'ratio': 50, 'loader': D2_loader, 'name': 'MVC-50'},
    {'ratio': 70, 'loader': D3_loader, 'name': 'MVC-70'},
]

for cfg in expert_configs:
    model = torchvision.models.resnet18(weights=None, num_classes=10).to(device)
    model = load_checkpoint(model, f'checkpoints/expso_expert_{cfg["ratio"]}.pth')
    clean = evaluate(model, testloader)
    print(f'Loaded Expert {cfg["ratio"]}%: Clean={clean:.2f}%')

Loaded Expert 30%: Clean=71.46%
Loaded Expert 50%: Clean=69.05%
Loaded Expert 70%: Clean=65.99%


In [4]:
# ==========================================================================
# PER-PARTITION ADVERSARIAL FINE-TUNING (Eq. 12)
# beta*_i = argmin_{beta_i} (1/|Di|) * sum L(f(x;beta_i), y)
# ==========================================================================
MVC_EPOCHS = 10
MVC_LR = 0.001

mvc_models = []

for cfg in expert_configs:
    print(f'\n{"="*70}')
    print(f'MVC TRAINING: {cfg["name"]} on Partition D{expert_configs.index(cfg)+1}')
    print(f'{"="*70}')

    # Load expert from Phase 2
    model = torchvision.models.resnet18(weights=None, num_classes=10).to(device)
    model = load_checkpoint(model, f'checkpoints/expso_expert_{cfg["ratio"]}.pth')

    # CRITICAL: Re-apply prune hooks to preserve sparsity during fine-tuning
    # Without this, SGD would overwrite the zero-valued (pruned) weights
    model = freeze_zeros(model)

    # Fine-tune on its specific partition with adversarial training
    optimizer = optim.SGD(model.parameters(), lr=MVC_LR, momentum=0.9, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MVC_EPOCHS)

    best_rob = 0
    for epoch in range(1, MVC_EPOCHS + 1):
        train_one_epoch(model, cfg['loader'], optimizer, nn.CrossEntropyLoss(), epoch,
            atk=torchattacks.PGD(model, eps=eps, alpha=alpha, steps=10))
        scheduler.step()

        if epoch % 5 == 0 or epoch == MVC_EPOCHS:
            rob = evaluate(model, testloader,
                atk=torchattacks.PGD(model, eps=eps, alpha=alpha, steps=20))
            clean = evaluate(model, testloader)
            sparsity = count_zero_params(model)
            print(f'Epoch {epoch}: Clean={clean:.2f}%, Robust(PGD-20)={rob:.2f}%, Sparsity={sparsity:.1f}%')
            if rob > best_rob:
                best_rob = rob
                fname = f'checkpoints/mvc_expert_{cfg["ratio"]}_v2.pth'
                save_checkpoint(model, fname)
                print(f'--> Saved {cfg["name"]}: {rob:.2f}%')

    mvc_models.append(model)
    del optimizer, scheduler
    gc.collect()
    torch.cuda.empty_cache()

print('\n--- MVC PER-PARTITION TRAINING COMPLETE ---')




MVC TRAINING: MVC-30 on Partition D1
  freeze_zeros: Locked 3,380,100/10,985,472 (30.8%) weights at zero


Epoch 5: 100%|██████████| 131/131 [01:15<00:00,  1.74it/s, loss=1.32, acc=47.6]


Epoch 5: Clean=71.40%, Robust(PGD-20)=39.07%, Sparsity=30.3%
--> Saved MVC-30: 39.07%


Epoch 10: 100%|██████████| 131/131 [01:15<00:00,  1.74it/s, loss=1.3, acc=48.3]


Epoch 10: Clean=71.71%, Robust(PGD-20)=38.42%, Sparsity=30.3%

MVC TRAINING: MVC-50 on Partition D2
  freeze_zeros: Locked 6,002,497/10,985,472 (54.6%) weights at zero


...

Epoch 10: Clean=69.32%, Robust(PGD-20)=38.44%, Sparsity=53.7%

MVC TRAINING: MVC-70 on Partition D3
  freeze_zeros: Locked 8,229,160/10,985,472 (74.9%) weights at zero


Epoch 5: 100%|██████████| 131/131 [00:29<00:00,  4.39it/s, loss=1.53, acc=41.3]


Epoch 5: Clean=65.55%, Robust(PGD-20)=37.50%, Sparsity=73.7%
--> Saved MVC-70: 37.50%


Epoch 10: 100%|██████████| 131/131 [00:29<00:00,  4.38it/s, loss=1.52, acc=41.7]


Epoch 10: Clean=65.39%, Robust(PGD-20)=37.53%, Sparsity=73.7%
--> Saved MVC-70: 37.53%

--- MVC PER-PARTITION TRAINING COMPLETE ---


In [5]:
# ==========================================================================
# MVC-NNT INFERENCE: RANDOM MODEL SELECTION
# ==========================================================================
# Reload best MVC models
mvc_models_best = []
for cfg in expert_configs:
    model = torchvision.models.resnet18(weights=None, num_classes=10).to(device)
    model = load_checkpoint(model, f'checkpoints/mvc_expert_{cfg["ratio"]}_v2.pth')
    model.eval()
    mvc_models_best.append(model)

# Create MVC-NNT ensemble with random selection
mvc_ensemble = MVC_NNT(mvc_models_best).to(device)

print('='*50)
print('MVC-NNT EVALUATION (Random Selection, 10 trials)')
print('='*50)

# Clean accuracy (random selection averaged over trials)
clean_mvc = mvc_ensemble.evaluate(testloader, device=str(device), n_trials=10)
print(f'Clean Accuracy: {clean_mvc:.2f}%')

# Robust accuracy: attack targets a random model, different model may predict
# This is the core defense: attacker doesn't know which model will be used
total_correct = 0
total_samples = 0
n_trials = 5
for trial in range(n_trials):
    for inputs, targets in testloader:
        inputs, targets = inputs.to(device), targets.to(device)
        # Attack a randomly chosen model
        atk_model = mvc_models_best[np.random.randint(0, 3)]
        atk = torchattacks.PGD(atk_model, eps=eps, alpha=alpha, steps=20)
        adv_inputs = atk(inputs, targets)
        # Predict with a DIFFERENT randomly chosen model
        with torch.no_grad():
            outputs = mvc_ensemble(adv_inputs)
            _, predicted = outputs.max(1)
            total_correct += predicted.eq(targets).sum().item()
            total_samples += targets.size(0)

robust_mvc = 100. * total_correct / total_samples
print(f'Robust Accuracy (PGD-20, random-model attack): {robust_mvc:.2f}%')

MVC-NNT EVALUATION (Random Selection, 10 trials)
Clean Accuracy: 68.60%
Robust Accuracy (PGD-20, random-model attack): 45.30%


In [6]:
# ==========================================================================
# GRADIENT COHERENCE ANALYSIS (Paper Fig. 11)
# Measures cosine similarity between gradient vectors of different models.
# Lower coherence = more diverse models = better adversarial defense.
# ==========================================================================
import torch.nn.functional as F_func

def compute_gradient_coherence(model1, model2, loader, device, n_batches=10):
    """Compute average cosine similarity between gradients of two models."""
    coherences = []
    criterion = nn.CrossEntropyLoss()

    for batch_idx, (inputs, targets) in enumerate(loader):
        if batch_idx >= n_batches:
            break
        inputs, targets = inputs.to(device), targets.to(device)

        # Get gradients from model 1
        model1.zero_grad()
        loss1 = criterion(model1(inputs), targets)
        loss1.backward()
        grads1 = torch.cat([p.grad.flatten() for p in model1.parameters() if p.grad is not None])

        # Get gradients from model 2
        model2.zero_grad()
        loss2 = criterion(model2(inputs), targets)
        loss2.backward()
        grads2 = torch.cat([p.grad.flatten() for p in model2.parameters() if p.grad is not None])

        # Truncate to same length (different pruning masks may cause size differences)
        min_len = min(len(grads1), len(grads2))
        cos_sim = F_func.cosine_similarity(grads1[:min_len].unsqueeze(0),
                                           grads2[:min_len].unsqueeze(0)).item()
        coherences.append(cos_sim)

    return np.mean(coherences)

# Enable gradients temporarily for coherence analysis
for m in mvc_models_best:
    m.train()
    for p in m.parameters():
        p.requires_grad = True

print('\nGradient Coherence (cosine similarity, lower = more diverse):')
print('-' * 40)
pairs = [(0, 1, '30% vs 50%'), (0, 2, '30% vs 70%'), (1, 2, '50% vs 70%')]
for i, j, label in pairs:
    coh = compute_gradient_coherence(mvc_models_best[i], mvc_models_best[j],
                                     testloader, str(device), n_batches=10)
    print(f'{label}: {coh:.4f}')

# Reset to eval mode
for m in mvc_models_best:
    m.eval()


Gradient Coherence (cosine similarity, lower = more diverse):
----------------------------------------
30% vs 50%: 0.1360
30% vs 70%: 0.0627
50% vs 70%: 0.0764
